In [1]:
import pandas as pd

In [11]:
return_average = pd.read_csv("grpo_return_total_average.csv", header = None)

In [12]:
return_average.rename(columns={0: "date", 1: "return"}, inplace=True)

In [48]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac

def newey_west_tstat(returns, maxlags=1):
    """
    논문 방식에 따른 Newey-West t-통계량 계산 함수
    입력:
        returns: 수익률 벡터 (list, np.array, pd.Series)
        maxlags: Newey-West 보정에 사용할 최대 시차
    출력:
        (평균 수익률, NW 표준오차, NW t-통계량)
    """
    returns = np.asarray(returns)
    T = len(returns)
    X = np.ones((T, 1))  # 상수항만 포함 (평균 추정)
    
    model = sm.OLS(returns, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    nw_cov = cov_hac(model, nlags=maxlags)
    # nw_se = np.sqrt(nw_cov[0, 0])
    # t_stat = model.params[0] / nw_se
    
    return model.params[0], model.bse[0], model.tvalues[0]


In [53]:

# 계산 실행
mean_return, nw_se, nw_tstat = newey_west_tstat(return_average, maxlags=1)


In [56]:
from scipy.stats import t



# 단측 검정 (우측): P(T > t)
p_value = 1 - t.cdf(nw_tstat, df=len(return_average))

print(f"p-value = {p_value:.6f}")


p-value = 0.000595


In [57]:
nw_tstat

np.float64(3.3680302506162714)

In [58]:
return_average

array([ 0.00173073,  0.02838954, -0.00698312,  0.01903733, -0.03351893,
        0.03543373, -0.00142419, -0.0427745 ,  0.01777361, -0.00253884,
        0.02420633, -0.00142655,  0.02288913, -0.00937845, -0.03474422,
       -0.02453808, -0.00955992,  0.04934951,  0.05213552,  0.07946993,
        0.0500288 , -0.01882056,  0.04941176,  0.03485664,  0.03170087,
        0.03674997,  0.09615742, -0.01256902,  0.05939766,  0.07185362,
       -0.06057905,  0.04649608,  0.01322407,  0.00405233,  0.10281413,
        0.01284564, -0.06036748,  0.02409889,  0.13038409,  0.01149406,
        0.04740475,  0.13936019,  0.04427982, -0.09400965,  0.0161644 ,
        0.03962058, -0.01969806, -0.00914615,  0.00700881, -0.05355759,
        0.07642353, -0.03136011, -0.04265661,  0.06695531,  0.01284897,
       -0.03506158,  0.00263852,  0.00468133,  0.0788759 , -0.02627443,
        0.03856898,  0.01971556, -0.03578013,  0.07767729,  0.25878548,
        0.1455471 ,  0.02976556,  0.01304267, -0.05752636,  0.00

In [59]:
np.quantile(return_average['return'], 0.05)

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices